In [ ]:
# ============================================
# California Cities Dataset
# K-Means Clustering + NLP
# ============================================

# 1. Import libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.cluster import KMeans
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA

# --------------------------------------------
# 2. Load the dataset
# --------------------------------------------

df = pd.read_csv("california_cities.csv")

# Display first rows
print(df.head())

# Dataset information
print("\nDataset shape:", df.shape)
print("\nColumns:")
print(df.columns.tolist())

print("\nMissing values:")
print(df.isnull().sum())


# ============================================
# PART A: K-MEANS ON NUMERICAL DATA
# ============================================

# 3. Select numerical columns
numeric_columns = df.select_dtypes(include=np.number).columns.tolist()

print("\nNumerical columns:")
print(numeric_columns)

# Remove columns that should not be used for clustering
# (for example, an ID/index column if one exists)
features = df[numeric_columns].copy()

# Replace missing values with the median
features = features.fillna(features.median())

# --------------------------------------------
# 4. Standardize the data
# --------------------------------------------

scaler = StandardScaler()
X_scaled = scaler.fit_transform(features)

print("\nScaled data shape:", X_scaled.shape)


# --------------------------------------------
# 5. Find a suitable number of clusters
#    using the Elbow Method
# --------------------------------------------

inertia = []
k_values = range(2, 11)

for k in k_values:
    kmeans = KMeans(
        n_clusters=k,
        random_state=42,
        n_init=10
    )
    kmeans.fit(X_scaled)
    inertia.append(kmeans.inertia_)

plt.figure(figsize=(8, 5))
plt.plot(k_values, inertia, marker="o")
plt.xlabel("Number of Clusters (K)")
plt.ylabel("Inertia")
plt.title("Elbow Method for K-Means")
plt.grid(True)
plt.show()


# --------------------------------------------
# 6. Apply K-Means
# --------------------------------------------

# Change this value after examining the elbow plot
K = 4

kmeans = KMeans(
    n_clusters=K,
    random_state=42,
    n_init=10
)

df["Cluster"] = kmeans.fit_predict(X_scaled)

print("\nCluster counts:")
print(df["Cluster"].value_counts().sort_index())


# --------------------------------------------
# 7. Display cluster information
# --------------------------------------------

print("\nCluster means:")
print(
    df.groupby("Cluster")[numeric_columns]
      .mean()
      .round(2)
)


# --------------------------------------------
# 8. Visualize clusters using PCA
# --------------------------------------------

pca = PCA(n_components=2, random_state=42)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(9, 6))

scatter = plt.scatter(
    X_pca[:, 0],
    X_pca[:, 1],
    c=df["Cluster"],
    cmap="viridis",
    s=50
)

plt.xlabel("Principal Component 1")
plt.ylabel("Principal Component 2")
plt.title("California Cities - K-Means Clusters")
plt.colorbar(scatter, label="Cluster")
plt.show()


# ============================================
# PART B: NLP USING CITY/TEXT INFORMATION
# ============================================

# 9. Find possible text columns
text_columns = df.select_dtypes(include=["object"]).columns.tolist()

print("\nText columns:")
print(text_columns)


# --------------------------------------------
# 10. Select a text column
# --------------------------------------------

# Usually a city/name column will contain city names.
# Change this if your CSV uses a different column name.

possible_city_columns = [
    col for col in text_columns
    if "city" in col.lower() or "name" in col.lower()
]

print("\nPossible city/name columns:")
print(possible_city_columns)


# Select the first matching column
if len(possible_city_columns) > 0:
    text_column = possible_city_columns[0]
else:
    text_column = text_columns[0]

print("\nUsing text column:", text_column)

text_data = df[text_column].fillna("").astype(str)


# --------------------------------------------
# 11. Convert text into TF-IDF features
# --------------------------------------------

vectorizer = TfidfVectorizer(
    lowercase=True,
    stop_words="english"
)

X_tfidf = vectorizer.fit_transform(text_data)

print("\nTF-IDF matrix shape:")
print(X_tfidf.shape)


# --------------------------------------------
# 12. K-Means clustering on NLP features
# --------------------------------------------

K_NLP = 4

kmeans_nlp = KMeans(
    n_clusters=K_NLP,
    random_state=42,
    n_init=10
)

df["NLP_Cluster"] = kmeans_nlp.fit_predict(X_tfidf)

print("\nNLP cluster counts:")
print(df["NLP_Cluster"].value_counts().sort_index())


# --------------------------------------------
# 13. Display cities in each NLP cluster
# --------------------------------------------

for cluster in sorted(df["NLP_Cluster"].unique()):
    print("\n==============================")
    print("NLP Cluster:", cluster)
    print("==============================")

    cities = df.loc[
        df["NLP_Cluster"] == cluster,
        text_column
    ]

    print(cities.to_string(index=False))


# ============================================
# PART C: SAVE RESULTS
# ============================================

# 14. Save clustered dataset

df.to_csv(
    "california_cities_clustered.csv",
    index=False
)

print("\nSaved as:")
print("california_cities_clustered.csv")